# DICOM

In [ ]:
import os
import numpy as np
import pydicom
import plotly.express as px

path_ = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"
read_dcm = lambda i : pydicom.dcmread(os.path.join(path_, i)).pixel_array
ct = list(map(read_dcm, sorted(os.listdir(path_))))
ct = np.stack(ct, axis=0)
print(ct.shape)

fig = px.imshow(
    ct, 
    animation_frame=0,
    binary_string=True, 
    labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
# Manufacturer (0008, 0070)
# X-Ray Tube Current (0018,1151)
# Exposure (0018,1152)
# Exposure Time (0018,1150)
# Exposure = (X-Ray Tube Current) * (Exposure Time)

import os, glob, random
import pydicom

def print_recursiverly(path, data):
    try:
        is_dcm = all([pydicom.misc.is_dicom(j) for j in glob.glob(os.path.join(path, "*"))])
    except PermissionError:
        is_dcm = False
    
    if is_dcm:
        if len(glob.glob(os.path.join(path, "*"))) > 0:
            dcm = pydicom.dcmread(glob.glob(os.path.join(path, "*"))[0])
            if dcm.get((0x0008, 0x0060)).value == "CT":
                manufacturer = dcm.get((0x0008, 0x0070)).value
                xray_curr = dcm.get((0x0018, 0x1151)).value
                try:
                    exposure_t = dcm.get((0x0018, 0x1150)).value
                except (AttributeError, TypeError):
                    exposure_t = None
                try:
                    exposure = dcm.get((0x0018, 0x1152)).value
                except AttributeError:
                    exposure = None
                
                print(path)
                print(manufacturer)
                print(xray_curr)
                print(exposure_t)
                print(exposure)
                print()
                data.append({"manufacturer": manufacturer, "xray_curr": xray_curr, "exposure_t": exposure_t, 
                             "exposure": exposure, "path": path})
    else:
        for j in glob.glob(os.path.join(path, "*")):
            print_recursiverly(j, data)

data = []
N = 20

patients = glob.glob(os.path.join(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data", "*"))
for patient in random.sample(patients, N):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i, data)

patients = glob.glob(os.path.join(r"C:\Users\bilel.guetarni\Desktop\data\TCIA\HNSCC-3DCT-RT\manifest-1549495779734\HNSCC-3DCT-RT", "*"))
for patient in random.sample(patients, N):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i, data)

print(len(data))

In [ ]:
df = pandas.DataFrame(data)
print(df)

idx = df.loc[pandas.isna(df["exposure_t"])].index
df.loc[idx, "exposure_t"] = (1000 * df.loc[idx, "exposure"]) / df.loc[idx, "xray_curr"]

idx = df.loc[pandas.isna(df["exposure"])].index
df.loc[idx, "exposure"] = df.loc[idx, "xray_curr"] * (df.loc[idx, "exposure_t"] / 1000)

print(df)

In [ ]:
for _, row in df.iterrows():
    print(row["path"])
    print(row)

In [ ]:
for _, row in df[df["manufacturer"] == "GE MEDICAL SYSTEMS"].iterrows():
    print(row["path"])

In [ ]:
import plotly.express as px

fig = px.scatter(df, x="xray_curr", y="exposure_t", color="manufacturer")
fig.show()

fig = px.scatter(df, x="xray_curr", y="exposure", color="manufacturer")
fig.show()

In [ ]:
import plotly.express as px

for i in df["manufacturer"].unique():
    print(i)
    fig = px.histogram(df[df["manufacturer"] == i], x="exposure_t", nbins=10)
    fig.update_xaxes(range=[-0, 1200])
    fig.show()

In [ ]:
import pandas
import plotly.express as px

for type in df["type"].unique():
    dff = df[df["type"] == type]
    dff = [{"type": k, "value": len(dff[dff["value"] == k])} for k in dff["value"].unique()]
    dff = pandas.DataFrame(dff)
    print(dff)

    fig = px.pie(dff, values='value', names='type')
    fig.show()

In [ ]:
for i in df["value"].unique():
    print(i)
    dff = df[df["value"] == i]
    for i, row in dff.iterrows():
        print(row["path"])

In [ ]:
from datetime import datetime

datetime_object = datetime.strptime(f[0x0008, 0x0012].value, '%Y%m%d')
print(datetime_object.day)

In [ ]:
import glob, os

for i in glob.glob(os.path.join(r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\004", "CT*")):
    print(i)
    for j in glob.glob(os.path.join(i, "*")):
        print(j)
        for k in glob.glob(os.path.join(j, "*"))[:1]:
            if pydicom.misc.is_dicom(k):
                dcm = pydicom.dcmread(k)
                print(datetime.strptime(dcm[0x0008, 0x0012].value, '%Y%m%d'))

In [ ]:
global XCURRENT
XCURRENT = []

def print_recursiverly(path):
    try:
        is_dcm = all([pydicom.misc.is_dicom(j) for j in glob.glob(os.path.join(path, "*"))])
    except PermissionError:
        is_dcm = False
    
    if is_dcm:
        if len(glob.glob(os.path.join(path, "*"))) > 0:
            dcm = pydicom.dcmread(glob.glob(os.path.join(path, "*"))[0])
            if dcm.get((0x0008, 0x0060)).value == "CT":
                type = "CBCT" if "CBCT" in os.path.split(path)[1] else "CT"
                value = dcm.get((0x0018, 0x1151)).value
                print(value)
                XCURRENT.append({"type": type, "value": value})
    else:
        for j in glob.glob(os.path.join(path, "*")):
            print_recursiverly(j)

for patient in glob.glob(os.path.join(r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data", "*")):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i)

In [ ]:
import pandas
import plotly.express as px
import plotly.figure_factory as ff

df = pandas.DataFrame(XCURRENT)

group_labels = ['CT', 'CBCT']
hist_data = [df[df["type"] == i]["value"] for i in group_labels]
fig = ff.create_distplot(hist_data, group_labels)
fig.show()

The tag (0020,0052) Frame of Reference UID is available in three different ways:
    (0020,0052) Frame of Reference UID
    (3006,0010) -> item -> (0020,0052) Frame of Reference UID
    (3006,0020) -> item -> (3006,0024) Referenced Frame of Reference UID

In [ ]:
import cv2
from dicom_class import CT, RTSTRUCT
import plotly.express as px

import numpy as np
from dicom_utils import fill_vol_ctrs

rtstruct = RTSTRUCT(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm")

ct = CT(
    path=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_CT_2017-07-17_133250_ORL.(sauf.sinus)_ORL.2MM.ARTIX_n200__00000",
    rtstruct=rtstruct
)

slide = 138
contour_name = "Encephale"

ctrs = ct.rtstruct.get_contours(contour_name)
mask = fill_vol_ctrs(ct.nii.shape, ctrs)
# print(mask.shape)

ct_img = ct.get_voxel_array()

src1 = np.moveaxis(ct_img, -1, 0)[slide]
src2 = np.moveaxis(mask.astype(ct_img.dtype), -1, 0)[slide]

src1 = ((src1 - np.min(src1)) / np.max(src1)) * 255
src2 = ((src2 - np.min(src2)) / np.max(src2)) * 255

img = cv2.addWeighted(src1, 0.5, src2, 0.5, 0.0)

fig = px.imshow(img)
fig.show()

# RTDOSE

In [ ]:
from dicom_class import CT, RTDOSE
from dicompylercore import dicomparser, dvh, dvhcalc

ct = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_CT_2017-07-17_133250_ORL.(sauf.sinus)_ORL.2MM.ARTIX_n200__00000"
dose = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402274.577.1575.dcm"
st = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm"

In [ ]:
# i.e. Get a dict of structure information
dp = dicomparser.DicomParser(st)
structures = dp.GetStructures()
structures

In [ ]:
# Calculate a DVH from DICOM RT data
calcdvh = dvhcalc.get_dvh(st, dose, 17)
# calcdvh.relative_volume.plot()
print(calcdvh.mean)

In [ ]:
dose1 = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402274.577.1575.dcm"
dose2 = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.5_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402276.1060.1577.dcm"
st = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm"

dp = dicomparser.DicomParser(st)
for _, str in dp.GetStructures().items():
    print(str["name"])
    calcdvh = dvhcalc.get_dvh(st, dose1, str["id"])
    print(calcdvh.mean)
    calcdvh = dvhcalc.get_dvh(st, dose2, str["id"])
    print(calcdvh.mean)

In [ ]:
from dicom_class import CT, RTDOSE, RTSTRUCT
import plotly.express as px
import numpy as np

dose = RTDOSE(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000")
rtstruct = RTSTRUCT(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm")
ct = CT(
    path=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_CT_2017-07-17_133250_ORL.(sauf.sinus)_ORL.2MM.ARTIX_n200__00000",
    rtdose=dose,
    rtstruct=rtstruct)


structure_name = "C_ext-3mm"

# mask = dose.get_structure_mask(structure_name)

# fig = px.imshow(
#     np.moveaxis(mask, -1, 0), 
#     animation_frame=0,
#     binary_string=True, 
#     labels=dict(animation_frame="slice"))
# fig.show()

In [ ]:
# fig = px.imshow(
#     np.moveaxis(dose.get_voxel_array(), -1, 0), 
#     animation_frame=0,
#     binary_string=True, 
#     labels=dict(animation_frame="slice"))

dose_array = dose.get_voxel_array()
fig = px.imshow(np.moveaxis(dose.get_voxel_array(), -1, 0)[40])
fig.show()

In [ ]:
structure_name = "Parotide_homolaterale"
dose_array = dose.get_voxel_array()
dose_array[dose.get_structure_mask(structure_name)].mean()

In [ ]:
from dicompylercore import dicomparser, dvh, dvhcalc
dp = dicomparser.DicomParser(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm")
dp.GetStructures()

In [ ]:
calcdvh = dvhcalc.get_dvh(
    r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm", 
    r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402274.577.1575.dcm", 
    17)
print(calcdvh.mean)

# clinical

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_MDA_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[1]]
print(df["USUBJID"].unique())
print(artix.mda_parsing(df))

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_SALIVATION_DATA_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[1]]
print(df["USUBJID"].unique())
print(artix.salivation_parsing(df))

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_DOSIMETRY_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[0]]
print(df["USUBJID"].unique())
print(artix.dosimetry_parsing(df))

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_AE_TOX_GEN_LTSI.csv", sep=";", encoding='cp1252')
df = df[df["USUBJID"] == df["USUBJID"].unique()[0]]
print(artix.aetox_parsing(df))

# print(df["USUBJID"].unique())
# print(len(df[df["USUBJID"].astype(int) == 5]))

# create ARTIX

In [ ]:
import glob, os, tqdm, random
import artix

path = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data"
patients = []
for p in tqdm.tqdm(glob.glob(os.path.join(path, "*"))[:3], ncols=50):
    patients.append(artix.load_patient(
        path=p,
        id_map=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\ARTIX_ID_CORRELATION.xlsx",
        PATIENT_DESCRIPTION_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_PATIENT_DESCRIPTION_LTSI.csv",
        EFFICACY_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_EFFICACY_LTSI.csv",
        SALIVATION_DATA_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_SALIVATION_DATA_LTSI.csv",
        TREATMENT_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_TREATMENT_LTSI.csv",
        DOSIMETRY_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_DOSIMETRY_LTSI.csv",
        MDA_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_MDA_LTSI.csv",
        AETOXGEN_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_AE_TOX_GEN_LTSI.csv",
        log="./log",
        ))

print(len(patients))

In [ ]:
import pickle

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix - Copie.pkl", "wb") as f:
    pickle.dump(patients, f)

## check segmentation

In [ ]:
import dicom_utils
import numpy as np

OAR_to_seg = {7: "parotid_gland_left", 8: "parotid_gland_right", 9: "submandibular_gland_right", 10: "submandibular_gland_left"}
threshold = 0.1

for p in patients:
    print(p.id)
    
    for ct in p.ct:
        print(ct.path)
        if ct.rtstruct is None:
            continue

        # TotalSegmentator must be applied before reloading nii
        out = ct.apply_totalsegmentator()

        ct.load_nii()

        for structure_ID, structure_name in OAR_to_seg.items():
            print(structure_name)
            struct_mask = (out == structure_ID).astype("uint8")

            d = 0
            best_oar = None
            for oar in ct.rtstruct.get_all_OARs():
                voxel_coords = ct.rtstruct.get_contours(oar)
                if voxel_coords:
                    original_mask = dicom_utils.fill_vol_ctrs(struct_mask.shape, voxel_coords)

                    dice = 2 * (np.logical_and(struct_mask, original_mask).sum()) / (struct_mask.sum() + original_mask.sum())
                    if dice > d:
                        d = dice
                        best_oar = oar
            print(f"{d:.2f} {best_oar}")

In [ ]:
import pydicom

for p in patients:
    for ct in p.ct:
        if ct.rtstruct is None:
            continue

        dcm = pydicom.dcmread(ct.rtstruct.get_dcm_path())
        for roi_contour in dcm.ROIContourSequence:
            if not hasattr(roi_contour, "ContourSequence"):
                print(f"[NO]\t {ct.rtstruct.path}")

# plots

In [ ]:
def is_float_try(str):
    try:
        float(str)
        return True
    except ValueError:
        return False

In [ ]:
import pickle
import artix

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix.pkl", "rb") as f:
# with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix - Copie.pkl", "rb") as f:
    patients = pickle.load(f)

print(len(patients))

In [ ]:
patients[0].clinical

In [ ]:
import re
import pandas

# SSF
df = []
for p in patients:
    df_p = {"id": p.id}
    for k, v in p.clinical.items():
        if not "salivation" in k:
            continue

        if not (isinstance(v, (int, float)) or is_float_try(v)) or pandas.isna(v):
            continue

        if "Inclusion".lower() in k.lower():
            df_p.update({"ssf_0": float(v)})
            continue

        M = re.findall("(M\d+)", k)
        if len(M) > 0:
            M = int(re.findall("\d+", M[0])[0])
            df_p.update({f"ssf_{M}": float(v)})

        df_p.update(p.clinical)
    
    df.append(df_p)
        
df = pandas.DataFrame(df)
df

In [ ]:
import plotly.graph_objects as go

dff = df[df["ssf_0"] >= 500]

k = "Standard Arm"
print(f"number of patients with xerostomia baseline in {k}: ", len(df[(df["ssf_0"] < 500) & (df["arm"] == k)]))
k = "Replanification Arm"
print(f"number of patients with xerostomia baseline in {k}: ", len(df[(df["ssf_0"] < 500) & (df["arm"] == k)]))

st6 = len(dff[(dff["ssf_6"] < 500) & (dff["arm"] == "Standard Arm")])
rp6 = len(dff[(dff["ssf_6"] < 500) & (dff["arm"] == "Replanification Arm")])

st12 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] < 500) & (dff["arm"] == "Standard Arm")])
rp12 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] < 500) & (dff["arm"] == "Replanification Arm")])

st18 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] < 500) & (dff["arm"] == "Standard Arm")])
rp18 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] < 500) & (dff["arm"] == "Replanification Arm")])

st24 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["ssf_24"] < 500) & (dff["arm"] == "Standard Arm")])
rp24 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["ssf_24"] < 500) & (dff["arm"] == "Replanification Arm")])


fig = go.Figure(data=[
    go.Bar(name="Standard Arm", x=[6, 12, 18, 24], y=[st6, st12, st18, st24]),
    go.Bar(name="Replanification Arm", x=[6, 12, 18, 24], y=[rp6, rp12, rp18, rp24])
])
# Change the bar mode
fig.update_layout(title="number of patients developing xerostomia throughout", barmode='group')
fig.show()



st6 = len(dff[(dff["ssf_6"] >= 500) & (dff["arm"] == "Standard Arm")])
rp6 = len(dff[(dff["ssf_6"] >= 500) & (dff["arm"] == "Replanification Arm")])

st12 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["arm"] == "Standard Arm")])
rp12 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["arm"] == "Replanification Arm")])

st18 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["arm"] == "Standard Arm")])
rp18 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["arm"] == "Replanification Arm")])

st24 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["ssf_24"] >= 500) & (dff["arm"] == "Standard Arm")])
rp24 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["ssf_24"] >= 500) & (dff["arm"] == "Replanification Arm")])


fig = go.Figure(data=[
    go.Bar(name="Standard Arm", x=[6, 12, 18, 24], y=[st6, st12, st18, st24]),
    go.Bar(name="Replanification Arm", x=[6, 12, 18, 24], y=[rp6, rp12, rp18, rp24])
])
# Change the bar mode
fig.update_layout(title="number of patients without xerostomia throughout", barmode='group')
fig.show()





st6 = len(dff[(dff["ssf_6"] >= 500) & (dff["arm"] == "Standard Arm")])
rp6 = len(dff[(dff["ssf_6"] >= 500) & (dff["arm"] == "Replanification Arm")])

st12 = len(dff[(dff["ssf_12"] >= 500) & (dff["arm"] == "Standard Arm")])
rp12 = len(dff[(dff["ssf_12"] >= 500) & (dff["arm"] == "Replanification Arm")])

st18 = len(dff[(dff["ssf_18"] >= 500) & (dff["arm"] == "Standard Arm")])
rp18 = len(dff[(dff["ssf_18"] >= 500) & (dff["arm"] == "Replanification Arm")])

st24 = len(dff[(dff["ssf_24"] >= 500) & (dff["arm"] == "Standard Arm")])
rp24 = len(dff[(dff["ssf_24"] >= 500) & (dff["arm"] == "Replanification Arm")])


fig = go.Figure(data=[
    go.Bar(name="Standard Arm", x=[6, 12, 18, 24], y=[st6, st12, st18, st24]),
    go.Bar(name="Replanification Arm", x=[6, 12, 18, 24], y=[rp6, rp12, rp18, rp24])
])
# Change the bar mode
fig.update_layout(title="number of patients with SSF>500 at any point", barmode='group')
fig.show()

In [ ]:
import plotly.graph_objects as go

no_xbase_6 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] < 500)])
xbase_6 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] < 500)])

no_xbase_12 = len(df[(df["ssf_0"] >= 500) & (df["ssf_12"] < 500)])
xbase_12 = len(df[(df["ssf_0"] < 500) & (df["ssf_12"] < 500)])

no_xbase_18 = len(df[(df["ssf_0"] >= 500) & (df["ssf_18"] < 500)])
xbase_18 = len(df[(df["ssf_0"] < 500) & (df["ssf_18"] < 500)])

no_xbase_24 = len(df[(df["ssf_0"] >= 500) & (df["ssf_24"] < 500)])
xbase_24 = len(df[(df["ssf_0"] < 500) & (df["ssf_24"] < 500)])


fig = go.Figure(data=[
    go.Bar(name="no xero. baseline", x=[6, 12, 18, 24], y=[no_xbase_6, no_xbase_12, no_xbase_18, no_xbase_24]),
    go.Bar(name="xero. baseline", x=[6, 12, 18, 24], y=[xbase_6, xbase_12, xbase_18, xbase_24])
])
# Change the bar mode
fig.update_layout(title="number of patients developing xerostomia throughout", barmode='group')
fig.show()





no_xbase_6 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] < 500)])
xbase_6 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] < 500)])

no_xbase_12 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] < 500)])
xbase_12 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] < 500)])

no_xbase_18 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] >= 500) & (df["ssf_18"] < 500)])
xbase_18 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] >= 500) & (df["ssf_18"] < 500)])

no_xbase_24 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] >= 500) & (df["ssf_18"] >= 500) & (df["ssf_24"] < 500)])
xbase_24 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] >= 500) & (df["ssf_18"] >= 500) & (df["ssf_24"] < 500)])


fig = go.Figure(data=[
    go.Bar(name="no xero. baseline", x=[6, 12, 18, 24], y=[no_xbase_6, no_xbase_12, no_xbase_18, no_xbase_24]),
    go.Bar(name="xero. baseline", x=[6, 12, 18, 24], y=[xbase_6, xbase_12, xbase_18, xbase_24])
])
# Change the bar mode
fig.update_layout(title="number of patients developing xerostomia throughout with filtering", barmode='group')
fig.show()

In [ ]:
print("total")
print(len(df))

print("SSF baseline unknown")
print(len(df[pandas.isna(df["ssf_0"])]))

print("No xerostomia baseline")
print(len(df[df["ssf_0"] >= 500]))

print("xerostomia baseline")
print(len(df[df["ssf_0"] < 500]))

In [ ]:
import re
import pandas

# SSF
df = []
for p in patients:
    for k, v in p.clinical.items():
        if not "salivation" in k:
            continue

        if not (isinstance(v, (int, float)) or is_float_try(v)):
            continue

        if "Inclusion".lower() in k.lower():
            df.append({"M": 0, "SSF": float(v), "id": p.id, "arm": p.clinical["arm"]})
            continue

        M = re.findall("(M\d+)", k)
        # print(k, M)
        if len(M) > 0:
            M = int(re.findall("\d+", M[0])[0])
            df.append({"M": M, "SSF": float(v), "id": p.id, "arm": p.clinical["arm"]})
        
df = pandas.DataFrame(df)
print(df)

In [ ]:
import plotly.express as px

fig = px.box(df, x="M", y="SSF", color="arm", title="Simulated salivarry flow in mg/min (M=months)")
fig.add_hline(y=500, line_dash="dot",
              annotation_text="xerostomia threshold", 
              annotation_position="bottom left")
fig.show()

In [ ]:
import pandas

map_names = {
    'PAROH_DOSE': ("ipsilateral", "parotid"),
    'PAROC_DOSE': ("contralateral", "parotid"),
    'SMAXH_DOSE': ("ipsilateral", "submandibular"),
    'SMAXC_DOSE': ("contralateral", "submandibular"),
    'MOUTH_DOSE': ("both", "mouth"),
 }

df = []
for p in patients:
    for k, v in p.clinical.items():
        if (not k in map_names.keys()) or (isinstance(v, str) and not is_float_try(v)):
            continue

        side, oar = map_names[k]
        df.append({"id": p.id, "arm": p.clinical["arm"], "side": side, "oar": oar, "dose": float(v)})
        
df = pandas.DataFrame(df)
print(df)

In [ ]:
import plotly.express as px
import numpy as np

fig = px.box(df, x="arm", y="dose", color="side", facet_col="oar")
fig.show()

In [ ]:
import pandas

df = []
for p in patients:
    for cm in p.clinical_measurements:
        df.append({"id": p.id, **cm})

df = pandas.DataFrame(df)
df

In [ ]:
print(df["type"].unique())

In [ ]:
def fn(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None

idx = (df["type"] == "SSF") & (df["visitID"] == 'Inclusion')
dff = df[idx]
dff["value"] = dff["value"].apply(fn)

print("total: ", len(dff))
print("xersotomia baseline unknown: ", len(dff[dff["value"].isna()]))
print("xerostomia baseline positive: ", len(dff[dff["value"] < 500]))
print("xersotomia baseline negative: ", len(dff[dff["value"] >= 500]))

In [ ]:
import plotly.express as px

def fn(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None

dff = df[df["type"] == "SSF"]
dff["value"] = dff["value"].apply(fn)

# exclude basline xerostomia
inclusion_idx =  (dff["visitID"] == "Inclusion")
noxerobaseline_idx = (dff[inclusion_idx]["value"].notna()) & (dff[inclusion_idx]["value"] >= 500)
dff = dff[dff["id"].isin(dff[inclusion_idx][noxerobaseline_idx]["id"].unique())]

no_xero_id = [id_ for id_ in dff["id"].unique() if len(dff[(dff["id"] == id_) & (dff["value"] < 500)]) == 0]
xero_id = [id_ for id_ in dff["id"].unique() if not(id_) in no_xero_id]

xero_df = dff[df["id"].isin(xero_id)]
print(len(xero_id))
fig = px.line(xero_df, x='visitID', y='value', color='id')
fig.add_hline(y=500, line_dash="dot")
fig.update_layout(
    width=2000,
    height=1000)
fig.show()

no_xero_df = dff[df["id"].isin(no_xero_id)]
print(len(no_xero_id))
fig = px.line(no_xero_df, x='visitID', y='value', color='id')
fig.add_hline(y=500, line_dash="dot")
fig.update_layout(
    width=2000,
    height=1000)
fig.show()

In [ ]:
import plotly.express as px

def fn(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None

dff = df[df["type"] == "SSF"]
dff["value"] = dff["value"].apply(fn)

fig = px.histogram(dff, x="value", facet_row ="visitID")
fig.update_layout(height=1000)
fig.show()

In [ ]:
import plotly.express as px

def fn(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None

dff = df[(df["type"] == "MDA") & (df["sample"] == "Q10")]
dff["value"] = dff["value"].apply(fn)

fig = px.histogram(dff, x="value", facet_row ="visitID")
fig.update_layout(height=1200)
fig.show()

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import re

def transform_SSF(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None
    
def transform_MDA(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None
    
def transform_AE(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None
    
# def display(dff, type1, type2):
#     data = []
#     for id_ in dff["id"].unique():
#         try:
#             data.append({type1: dff[(dff["id"] == id_) & (dff["type"] == type1)]["value"].item()})
#         except ValueError:
#             data.append({type1: None})
        
#         try:
#             data.append({type2: dff[(dff["id"] == id_) & (dff["type"] == type2)]["value"].item()})
#         except ValueError:
#             data.append({type2: None})

#     data = pandas.DataFrame(data).astype("float")
#     print(data)
#     x = data[type1].unique()
#     y = data[type2].unique()
#     z = []
#     for t1 in x:
#         xy = []
#         for t2 in y:
#             xy.append(len(data[(data[type1] == t1) & (data[type2] == t2)]))
#         z.append(xy)

#     # fig.update_layout(width=800, height=1000)
#     fig = go.Figure(data=go.Heatmap(z=z, x=x, y=y, hoverongaps = True))
#     fig.show()

# select xerostomia CTCAE and MDA
dff = df[((df["type"] == "MDA") & (df["sample"] == "Q10")) | ((df["type"] == "AE") & (df["sample"] == "XEROSTOMIE")) | (df["type"] == "SSF")]

# transform values

SSF_idx = (dff["type"] == "SSF")
dff.loc[SSF_idx, "value"] = dff[SSF_idx]["value"].apply(transform_SSF)

MDA_idx = (dff["type"] == "MDA")
dff.loc[MDA_idx,"value"] = dff[MDA_idx]["value"].apply(transform_MDA)

AE_idx = (dff["type"] == "AE")
dff.loc[AE_idx,"value"] = dff[AE_idx]["value"].apply(transform_AE)

# change AE visitID=S0 into Inclusion
dff.loc[(dff["type"] == "AE") & (dff["visitID"] == "S0"), "visitID"] = "Inclusion"

In [ ]:
t1, t2 = "AE", "MDA"

x = sorted(dff[dff["type"] == t1]["value"].dropna().unique())
y = sorted(dff[dff["type"] == t2]["value"].dropna().unique())

data = []
for xx in x:
    counts = []
    for yy in y:
        counts.append(0)
        for visit in dff["visitID"].unique():
            for id in dff["id"].unique():
                sub_dff = dff[(dff["type"].isin([t1, t2])) & (dff["visitID"] == visit) & (dff["id"] == id)]
                try:
                    if sub_dff[sub_dff["type"] == t1]["value"].item() == xx and sub_dff[sub_dff["type"] == t2]["value"].item() == yy:
                        counts[-1] = counts[-1] + 1 
                except ValueError:
                    continue
    data.append(counts)

fig = px.imshow(data,
                # labels=dict(x="Day of Week", y="Time of Day", color="Productivity"),
                x=list(map(str, y)),
                y=list(map(str, x)),
                    text_auto=True
            )
fig.show()

In [ ]:
sub_dff = []
for visit in dff["visitID"].unique():
    for id in dff["id"].unique():
        sub_dff.append({})
        for t in dff["type"].unique():
            try:
                sub_dff[-1].update({t: dff[(dff["type"] == t) & (dff["visitID"] == visit) & (dff["id"] == id)]["value"].item()})
            except ValueError:
                sub_dff[-1].update({t: None})

sub_dff = pandas.DataFrame(sub_dff)

fig = px.histogram(sub_dff, x="SSF", facet_row ="AE", marginal='violon')
fig.update_layout(height=1000)
fig.show()

sub_dff.loc[sub_dff["MDA"].isin([0,1,2]), "MDA"] = 1
sub_dff.loc[sub_dff["MDA"].isin([3,4,5]), "MDA"] = 2
sub_dff.loc[sub_dff["MDA"].isin([6,7,8]), "MDA"] = 3
sub_dff.loc[sub_dff["MDA"].isin([9]), "MDA"] = 4
fig = px.histogram(sub_dff, x="SSF", facet_row ="MDA", marginal='violon')
fig.update_layout(height=1000)
fig.show()

In [ ]:
import pandas
import pickle
import re

import artix


def transform_SSF(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None
    
def transform_MDA(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None
    
def transform_AE(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix.pkl", "rb") as f:
    patients = pickle.load(f)

print(len(patients))

df = []
for p in patients:
    for cm in p.clinical_measurements:
        df.append({"id": p.id, **cm})

df = pandas.DataFrame(df)
print(df.columns)

# select xerostomia CTCAE and MDA
dff = df[((df["type"] == "MDA") & (df["sample"] == "Q10")) | ((df["type"] == "AE") & (df["sample"] == "XEROSTOMIE")) | (df["type"] == "SSF")]

# transform values

SSF_idx = (dff["type"] == "SSF")
dff.loc[SSF_idx, "value"] = dff[SSF_idx]["value"].apply(transform_SSF)

MDA_idx = (dff["type"] == "MDA")
dff.loc[MDA_idx,"value"] = dff[MDA_idx]["value"].apply(transform_MDA)

AE_idx = (dff["type"] == "AE")
dff.loc[AE_idx,"value"] = dff[AE_idx]["value"].apply(transform_AE)

# change AE visitID=S0 into Inclusion
dff.loc[(dff["type"] == "AE") & (dff["visitID"] == "S0"), "visitID"] = "Inclusion"

print(dff)

In [ ]:
dff[dff["type"] == "AE"]["value"].unique()

In [ ]:
from scipy.stats import f_oneway, spearmanr

sub_dff = []
for visit in dff["visitID"].unique():
    for id in dff["id"].unique():
        sub_dff.append({})
        for t in dff["type"].unique():
            try:
                sub_dff[-1].update({t: dff[(dff["type"] == t) & (dff["visitID"] == visit) & (dff["id"] == id)]["value"].item()})
            except ValueError:
                sub_dff[-1].update({t: None})

sub_dff = pandas.DataFrame(sub_dff)

# # keep only rows were all ifnor is available
# sub_dff = sub_dff[(sub_dff["SSF"].notna()) & (sub_dff["MDA"].notna()) & (sub_dff["AE"].notna())]

# group MDA into smaller groups
sub_dff.loc[sub_dff["MDA"].isin([0,1,2]), "MDA"] = 1
sub_dff.loc[sub_dff["MDA"].isin([3,4,5]), "MDA"] = 2
sub_dff.loc[sub_dff["MDA"].isin([6,7,8]), "MDA"] = 3
sub_dff.loc[sub_dff["MDA"].isin([9]), "MDA"] = 4


sub_dff_bis = sub_dff[(sub_dff["SSF"].notna()) & (sub_dff["AE"].notna())]
groups = [sub_dff_bis[sub_dff_bis["AE"] == i]["SSF"].to_numpy() for i in sub_dff_bis["AE"].unique()]
print(len(groups))
print(f_oneway(*groups))
print(spearmanr(sub_dff_bis["SSF"].to_numpy(), sub_dff_bis["AE"].to_numpy()))

sub_dff_bis = sub_dff[(sub_dff["SSF"].notna()) & (sub_dff["MDA"].notna())]
groups = [sub_dff_bis[sub_dff_bis["MDA"] == i]["SSF"].to_numpy() for i in sub_dff_bis["MDA"].unique()]
print()
print(len(groups))
print(f_oneway(*groups))
print(spearmanr(sub_dff_bis["SSF"].to_numpy(), sub_dff_bis["MDA"].to_numpy()))

print()
sub_dff_bis = sub_dff[(sub_dff["AE"].notna()) & (sub_dff["MDA"].notna())]
print(spearmanr(sub_dff_bis["MDA"].to_numpy(), sub_dff_bis["AE"].to_numpy()))